# 阶段 4-1：训练 3-8-3 MLP

输入：`data/features.csv`（阶段 3 真机采集）。
如果没有真机数据，会自动改用 `data/features_demo.csv`（`make_demo_csv.py` 造的示范数据），
这样整条链路可以先在 PC 上跑通。

输出：`data/mlp_float.npz`（训练好的浮点权重，阶段 4-2 量化用）。

In [ ]:
import os
import sys
import numpy as np

# 自动定位工程根目录：从当前目录往上找，直到看见 lib/video_setup.py。
# 这样 ai_video 无论放在 /home/xilinx/jupyter_notebooks/ai_video，
# 还是塞在 OV5640_capture_only/ai_video 里，notebook 都不用改路径。
ROOT = os.path.abspath(os.getcwd())
while ROOT != os.path.dirname(ROOT) and not os.path.isfile(os.path.join(ROOT, "lib", "video_setup.py")):
    ROOT = os.path.dirname(ROOT)
STAGE = os.path.join(ROOT, "stage04_mlp_train")
sys.path.insert(0, ROOT)
sys.path.insert(0, STAGE)

import mlp_common as mc
from lib import features_io

csv_path, kind = features_io.pick_features_csv(ROOT)
print("数据来源:", csv_path, "(%s)" % kind)
if kind == "demo":
    print("!! 这是示范数据，不是真机采集；最终提交前请用阶段 3 的真机 features.csv 重训")

X, y = features_io.load_csv(csv_path)
print("样本数:", len(X), " 特征维度:", X.shape[1], " 类别:", sorted(set(y.tolist())))

# ---- 数据体检 --------------------------------------------------------------
# 看两件事：
#   1) 每个特征在每个类别里的 均值 +- 标准差（数据长什么样）
#   2) 逐特征的类间可分性：两类相差几个标准差（这个才决定"能不能训"）
#
# 重要：某个特征在某一类里 std=0 不是错误。
#   画面没动（静态场景）、或者这个量本身就很稳，标准差自然就是 0。
#   真正要报警的是"某个特征在三类之间是同一个值"——那才是被移位截断、彻底没信息。
#   （第一版 f_grad 定成 >>20，三类全是 2.00+-0.00，就是这个情况。）
cols = list(features_io.FEATURE_COLUMNS)
klasses = sorted(set(y.tolist()))

print("")
print("逐特征 / 逐类别 统计（均值 +- 标准差）:")
for c, name in enumerate(cols):
    line = "  %-8s" % name
    for k in klasses:
        v = X[y == k, c]
        line += "  label%d: %8.2f+-%7.2f" % (k, v.mean(), v.std())
    print(line)

print("")
print("逐特征的类间可分性（差值 / 合并标准差；>=3 就算分得开）:")
flat_features = []
for c, name in enumerate(cols):
    means = [X[y == k, c].mean() for k in klasses]
    stds = [max(X[y == k, c].std(), 1e-9) for k in klasses]
    if max(means) - min(means) < 1e-9:
        flat_features.append(name)
    line = "  %-8s" % name
    for a in range(len(klasses)):
        for b in range(a + 1, len(klasses)):
            z = abs(means[a] - means[b]) / ((stds[a] ** 2 + stds[b] ** 2) ** 0.5)
            line += "  %d-%d:%6.1f" % (klasses[a], klasses[b], z)
    print(line)

# 每一对类别，只要在某个特征上能拉开 >=3σ，就说明这一对可分
pair_best = {}
for c, name in enumerate(cols):
    means = [X[y == k, c].mean() for k in klasses]
    stds = [max(X[y == k, c].std(), 1e-9) for k in klasses]
    for a in range(len(klasses)):
        for b in range(a + 1, len(klasses)):
            z = abs(means[a] - means[b]) / ((stds[a] ** 2 + stds[b] ** 2) ** 0.5)
            key = (klasses[a], klasses[b])
            if z > pair_best.get(key, -1):
                pair_best[key] = z

print("")
print("每一对类别的最好可分性:")
all_ok = True
for key in sorted(pair_best):
    z = pair_best[key]
    flag = "OK" if z >= 3 else "!! 分不开"
    if z < 3:
        all_ok = False
    print("   类别 %d vs %d : 最好 %5.1fσ  %s" % (key[0], key[1], z, flag))

if flat_features:
    print("")
    print("!! 报警：这些特征在三类之间均值完全相同，等于没接进来：", flat_features)
    print("   回阶段 3 检查是不是移位定得太粗、把特征截断了（见 stage03 的 verify.md）")
elif not all_ok:
    print("")
    print("!! 报警：有类别对在三个特征上都分不开，MLP 训不出来。回阶段 3 重采更拉开的场景。")
else:
    print("")
    print("数据体检通过：三类两两都能分开（每一对至少在某一维上差 3σ 以上）。")
    print("个别特征在某一类里 std=0 是正常的：静态场景本来就不会变。")


## 训练

这里用纯 numpy 实现（原因见 `mlp_common.py` 开头）：训练时做标准化，
训练结束后把标准化**折进第一层权重**，所以定点推理只吃原始特征。

In [ ]:
# ---- 留 20% 做验证集 --------------------------------------------------------
# 为什么必须留：三类差别很大，用全部数据训练拿到 100% 是必然的，
# 说明不了泛化能力。留一份从没参与训练的数据，报出来的准确率才经得起问。
rng = np.random.RandomState(0)
perm = rng.permutation(len(X))
n_val = max(1, int(round(len(X) * 0.2)))
val_idx, tr_idx = perm[:n_val], perm[n_val:]
print("训练集 %d 条 / 验证集 %d 条" % (len(tr_idx), len(val_idx)))

# 训练 + 量化一次到位：自动挑一组「准确率够、量化又不饱和」的超参。
# 直接用 mc.train(lr=0.5) 的话，类内变化少（比如某类只有几个不同取值）时
# 折进第一层的权重会超过 0.5，阶段 4-2 的 quantize() 就会拒绝导出。
model, _qmodel, q_info = mc.train_for_quantization(X[tr_idx], y[tr_idx], verbose=True)

acc_tr = float((mc.predict_float(model, X[tr_idx]) == y[tr_idx]).mean())
acc_va = float((mc.predict_float(model, X[val_idx]) == y[val_idx]).mean())
print("训练集准确率: %.4f" % acc_tr)
print("验证集准确率: %.4f    <-- 报告/答辩里写这个数" % acc_va)

pred = mc.predict_float(model, X)
print("全量数据准确率: %.4f" % float((pred == y).mean()))

classes = sorted(set(y.tolist()))
print("混淆矩阵 (行=真实, 列=预测):")
cm = np.zeros((len(classes), len(classes)), dtype=int)
for t, p in zip(y, pred):
    cm[classes.index(t), classes.index(p)] += 1
print("      " + "".join("pred%d " % c for c in classes))
for i, c in enumerate(classes):
    print("真%d  " % c + "".join("%6d" % v for v in cm[i]))

assert acc_va > 0.85, "验证集准确率不足 85%：先回阶段 3 检查三类场景是否真的拉开"

out = os.path.join(ROOT, "data", "mlp_float.npz")
np.savez(out,
         w1=model["w1"], b1=model["b1"],
         w2=model["w2"], b2=model["b2"],
         mean=model["mean"], std=model["std"],
         classes=model["classes"])
print("已保存浮点模型:", out)


## 下一步

准确率 > 85% 后进入 `04_mlp_quant_export.ipynb` 做 Q8.8 量化并导出 `mlp_weights.h`。

## 数据体检（和 tools/check_features.py 同一套判据）

这一步放在训练旁边，是为了**训练前就发现数据的问题**：训练本身跑得通，
但只在几个点上可靠的模型，实机落在中间地带就会乱判（实测 `f_luma=30` 被判成类别 1）。

两条最容易被忽略的判据：

* **C5 类间真空**：两类之间一大段没有样本 = 实机落进去判决任意；
* **C8 类内覆盖**：某一维在类内几乎不变（例如常规类的 `f_grad` 只有 2 个取值）= 模型把它当身份证，板上一漂判决就跳。

有 FAIL 就先补采那一类；WARN 可以带着走，但要在报告里说明。

In [ ]:
# 数据体检：直接调 tools/check_features.py，保证和采集端看到的是同一套判据。
import subprocess

r = subprocess.run([sys.executable, os.path.join(ROOT, "tools", "check_features.py")])
print("check_features 退出码 =", r.returncode)
if r.returncode != 0:
    print("!! 数据体检有 FAIL：按上面的提示补采那一类，再回来训练")
    print("   （训练照样能跑出 100% 训练准确率，但模型只在训练过的几个点上可靠）")